# Tổng hợp THỬ kết quả đã train (chưa đủ seed / chưa chạy sau_train)
Đọc mọi file `*.zip` trong thư mục kết quả (mỗi zip = 1 lượt, do `mot_luot.sh` tạo), **chỉ lấy**
`metrics.json` và `per_class_AP_test.csv` (không giải nén trọng số). Kết quả:
- in bảng mAP chuẩn COCO trên tập test (pycocotools) của từng lượt;
- kiểm tra mọi lượt dùng CÙNG dấu vân tay dữ liệu và cùng mã cấu hình;
- đánh dấu lượt có lật ngang hoặc không rõ (code cũ) → KHÔNG dùng cho bài;
- ghi `ket_qua_thu/` và nén `ket_qua_thu.zip` (vài trăm KB) để gửi Claude.

Chưa có: F1 (cần dự đoán tập valid + ngưỡng, do `sau_train` tạo), tốc độ, mean ± std (cần đủ 3 seed).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===== SỬA ĐƯỜNG DẪN NÀY nếu thư mục nằm chỗ khác (chuột phải thư mục ở cột trái → Copy path) =====
THU_MUC = '/content/drive/MyDrive/Kết quả H100'
OUT = '/content/ket_qua_thu'
ZIP_OUT = '/content/ket_qua_thu'          # → /content/ket_qua_thu.zip

In [3]:
import csv
import io
import json
import shutil
import zipfile
from pathlib import Path

src = Path(THU_MUC)
assert src.is_dir(), f'Không thấy thư mục {src}. Sửa THU_MUC ở ô trên.'
zips = sorted(z for z in src.glob('*.zip') if z.name != 'ket_qua_thu.zip')
print(f'Tìm thấy {len(zips)} file zip:'); [print('  ', z.name) for z in zips]
out = Path(OUT); shutil.rmtree(out, ignore_errors=True); (out / 'per_class').mkdir(parents=True)

rows, per_class, loi = [], {}, []
for z in zips:
    try:
        zf = zipfile.ZipFile(z)
    except zipfile.BadZipFile:
        loi.append(f'{z.name}: zip hỏng / tải chưa xong'); continue
    mets = [n for n in zf.namelist() if n.endswith('/metrics.json') and '/_smoke/' not in n]
    if len(mets) != 1:
        loi.append(f'{z.name}: có {len(mets)} metrics.json (cần đúng 1)'); continue
    m = json.loads(zf.read(mets[0]))
    run_dir = mets[0].rsplit('/', 1)[0]
    c = m.get('coco_test', {})
    ln = m.get('lat_ngang', 'không ghi')          # code cũ không ghi khóa này
    if m.get('model', '').startswith(('rtdetr', 'rfdetr')):
        dung_duoc = (ln is False)
    else:
        dung_duoc = True                             # YOLO, Faster R-CNN: tắt lật ngang từ đầu
    rows.append({'file_zip': z.name, 'thu_muc_trong_zip': run_dir, 'dataset': m.get('dataset'),
                 'model': m.get('model'), 'seed': m.get('seed'), 'gpu': m.get('gpu'),
                 'mAP50_95': c.get('mAP50_95'), 'mAP50': c.get('mAP50'), 'mAP75': c.get('mAP75'),
                 'AP_small': c.get('AP_small'), 'AP_medium': c.get('AP_medium'), 'AP_large': c.get('AP_large'),
                 'lat_ngang': ln, 'dung_cho_bai': dung_duoc,
                 'config_sha256': m.get('config_sha256'), 'dataset_fingerprint': m.get('dataset_fingerprint'),
                 'train_seconds': m.get('train_seconds_last_session', m.get('train_seconds'))})
    (out / 'metrics').mkdir(exist_ok=True)
    (out / 'metrics' / (z.stem + '.json')).write_bytes(zf.read(mets[0]))
    pc = run_dir + '/per_class_AP_test.csv'
    if pc in zf.namelist():
        data = zf.read(pc).decode('utf-8-sig')
        (out / 'per_class' / (z.stem + '.csv')).write_text(data, encoding='utf-8-sig')
        per_class[z.stem] = {r['class_name']: r['AP50_95'] for r in csv.DictReader(io.StringIO(data))}
    log = [n for n in zf.namelist() if n.startswith('logs_h100/') and n.endswith('.log')]
    if log:
        (out / 'log_cuoi').mkdir(exist_ok=True)
        tail = zf.read(log[0]).decode('utf-8', 'replace').splitlines()[-60:]
        (out / 'log_cuoi' / (z.stem + '.txt')).write_text('\n'.join(tail), encoding='utf-8')

Tìm thấy 9 file zip:
   tda_frcnn_r50fpn_seed0.zip
   tda_rfdetr_seed0(có lật ngang).zip
   tda_rtdetr_seed0(có lật ngang).zip
   tda_yolo11n_seed0.zip
   tda_yolo11s_seed0.zip
   tda_yolo26n_seed0.zip
   tda_yolo26s_seed0.zip
   tda_yolov8n_seed0.zip
   tda_yolov8s_seed0.zip


In [4]:
def pct(v):
    return '   -  ' if v is None else f'{100 * float(v):6.2f}'

print(f'\n{"model":<22}{"ds":<6}{"seed":>4} {"mAP":>6} {"AP50":>6} {"AP75":>6} {"APs":>6} {"APm":>6} {"APl":>6}  lật ngang   dùng cho bài')
for r in sorted(rows, key=lambda r: (str(r['dataset']), -(r['mAP50_95'] or 0))):
    print(f'{str(r["model"]):<22}{str(r["dataset"])[:5]:<6}{str(r["seed"]):>4} {pct(r["mAP50_95"])} {pct(r["mAP50"])} {pct(r["mAP75"])} '
          f'{pct(r["AP_small"])} {pct(r["AP_medium"])} {pct(r["AP_large"])}  {str(r["lat_ngang"]):<10} {"có" if r["dung_cho_bai"] else "KHÔNG"}')

print('\nKiểm tra tính nhất quán:')
for k in ['dataset_fingerprint', 'config_sha256']:
    for ds in sorted({str(r['dataset']) for r in rows}):
        vals = {r[k] for r in rows if str(r['dataset']) == ds}
        print(f'  {ds} – {k}: {"✅ một giá trị " + str(next(iter(vals))) if len(vals) == 1 else "❌ KHÁC NHAU: " + str(vals)}')
for e in loi:
    print('  ❌', e)


model                 ds    seed    mAP   AP50   AP75    APs    APm    APl  lật ngang   dùng cho bài
rtdetrv2_r18vd        split    0  71.70  89.58  85.11  61.31  74.72  82.87  không ghi  KHÔNG
yolo26s               split    0  69.47  87.46  81.90  58.60  71.90  81.34  không ghi  có
yolov8s               split    0  69.17  88.12  82.46  57.93  73.50  79.53  không ghi  có
yolo11s               split    0  68.81  86.74  81.57  56.13  73.00  79.58  không ghi  có
yolov8n               split    0  66.80  85.40  78.45  49.00  70.88  81.90  không ghi  có
yolo11n               split    0  65.97  83.72  77.93  46.67  70.63  80.77  không ghi  có
yolo26n               split    0  65.71  83.33  77.62  51.05  68.99  81.61  không ghi  có
rfdetr_s              split    0  64.40  84.36  74.30  46.40  67.68  82.32  không ghi  KHÔNG
frcnn_r50fpn          split    0  59.67  78.84  72.11  39.34  61.88  79.96  không ghi  có

Kiểm tra tính nhất quán:
  split_data_by_box – dataset_fingerprint: ✅ một giá trị

In [5]:
with open(out / 'bang_tom_tat.csv', 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
lop = sorted({c for d in per_class.values() for c in d})
with open(out / 'AP_tung_lop_gop.csv', 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f); w.writerow(['class_name'] + list(per_class))
    for c in lop:
        w.writerow([c] + [per_class[k].get(c, '') for k in per_class])
(out / 'loi.txt').write_text('\n'.join(loi) or 'không có', encoding='utf-8')
shutil.make_archive(ZIP_OUT, 'zip', out)
shutil.copy(ZIP_OUT + '.zip', src / 'ket_qua_thu.zip')
print(f'\n✅ Đã ghi {src / "ket_qua_thu.zip"} — gửi file này cho Claude.')


✅ Đã ghi /content/drive/MyDrive/Kết quả H100/ket_qua_thu.zip — gửi file này cho Claude.
